# Full Pinceaux Analysis Runner
Use this notebook to run the full pipeline (edge correction -> MLI-only -> contours + surface area + volume).

This runner asks for:
- Pinceaux ID (X in `pinceaux_X`)
- Scale bar: length in um and pixels
- Slice thickness (nm)
- First and last z coordinates
- Whether screenshots were taken in ascending or descending z order

If a field is left blank in CLI usage, the pipeline falls back to `Inputs/Raw/pinceaux_X/analysis_config.json`.
This notebook run writes that config automatically so CI can run later from raw inputs only.

Processed slice filenames are written as `z_<index>_<z>.png`.

In [1]:
from pathlib import Path
import traceback

import ipywidgets as widgets
from IPython.display import display

from full_pipeline import run_full_pipeline

In [ ]:
base_dir = Path.cwd()

id_widget = widgets.IntText(value=5, description='Pinceaux ID:')
scale_um_widget = widgets.FloatText(value=2.0, description='Scale um:')
scale_px_widget = widgets.FloatText(value=126.0, description='Scale px:')
thickness_widget = widgets.FloatText(value=40.0, description='Slice nm:')
z_first_widget = widgets.FloatText(value=0.0, description='First z:')
z_last_widget = widgets.FloatText(value=0.0, description='Last z:')
order_widget = widgets.Dropdown(options=['ascending', 'descending'], value='ascending', description='Order:')
write_cfg_widget = widgets.Checkbox(value=True, description='Write raw config JSON')

run_button = widgets.Button(description='Run Full Analysis', button_style='success')
out = widgets.Output()

def on_run_clicked(_):
    with out:
        out.clear_output()
        try:
            print('Starting full analysis...')
            run_full_pipeline(
                base_dir=base_dir,
                pinceaux_id=id_widget.value,
                scale_um=scale_um_widget.value,
                scale_px=scale_px_widget.value,
                slice_thickness_nm=thickness_widget.value,
                z_first=z_first_widget.value,
                z_last=z_last_widget.value,
                capture_order=order_widget.value,
                write_config=write_cfg_widget.value,
            )
            print('Finished.')
        except Exception as exc:
            print(f'ERROR: {exc}')
            traceback.print_exc()

run_button.on_click(on_run_clicked)

display(widgets.VBox([
    id_widget,
    scale_um_widget,
    scale_px_widget,
    thickness_widget,
    z_first_widget,
    z_last_widget,
    order_widget,
    write_cfg_widget,
    run_button,
    out,
]))

## CLI alternative
You can run the same pipeline from terminal:

`python full_pipeline.py --id 5 --scale-um 2 --scale-px 126 --slice-thickness-nm 40 --z-first <z0> --z-last <zN> --capture-order ascending --write-config`

To run using only saved raw config:
`python full_pipeline.py --id 5`